In [1]:
# -*- coding: utf-8 -*-
"""
EDA для датасета банковских продуктов (устойчивый к ' NA' с пробелами)
"""

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
from matplotlib.backends.backend_pdf import PdfPages
import warnings
warnings.filterwarnings('ignore')

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

print("="*70)
print("ЗАГРУЗКА ДАННЫХ (СЕМПЛ 500 000 СТРОК)")
print("="*70)

# 1. Загрузим только заголовки, чтобы определить целевые колонки
sample_headers = pd.read_csv('data/train_ver2.csv', nrows=0)
target_cols = [col for col in sample_headers.columns if col.startswith('ind_') and col.endswith('_ult1')]
print(f"Найдено целевых колонок (продуктов): {len(target_cols)}")

# 2. Загрузим семпл данных (первые 500k строк, этого достаточно для EDA)
#    Если нужно больше, увеличьте nrows, но учтите память
nrows = 500_000  # можно поставить 1_000_000, если RAM > 8GB

# Расширенный список значений NA (включая пробельные варианты)
na_values = [' NA', '     NA', 'NA', 'n/a', 'N/A', 'NULL', 'null', '', ' ', '?', 'unknown']

# Читаем без указания dtype, пусть pandas сам определит типы
df = pd.read_csv('data/train_ver2.csv',
                 nrows=nrows,
                 na_values=na_values,
                 skipinitialspace=True,   # обрезает пробелы в начале и конце
                 low_memory=False)

print(f"Загружено {df.shape[0]:,} строк, {df.shape[1]} колонок")
print(f"Используемая память: {df.memory_usage(deep=True).sum() / 1024**2:.1f} MB")

# 3. Приводим целевые колонки к int8 (заполняем NaN нулями)
for col in target_cols:
    # Если колонка ещё не числовая, преобразуем
    df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0).astype('int8')

# 4. Преобразуем даты
df['fecha_dato'] = pd.to_datetime(df['fecha_dato'], errors='coerce')
if 'fecha_alta' in df.columns:
    df['fecha_alta'] = pd.to_datetime(df['fecha_alta'], errors='coerce')

# 5. Категориальные колонки -> category
cat_cols = ['ind_empleado', 'pais_residencia', 'sexo', 'indrel_1mes', 'tiprel_1mes',
            'indresi', 'indext', 'conyuemp', 'canal_entrada', 'indfall', 'nomprov', 'segmento']
for col in cat_cols:
    if col in df.columns:
        df[col] = df[col].astype('category')

# 6. Числовые колонки приводим к float32 (если есть некорректные значения -> NaN)
num_cols = ['age', 'antiguedad', 'renta', 'ind_nuevo', 'indrel', 'tipodom', 'cod_prov', 'ind_actividad_cliente']
for col in num_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce').astype('float32')

print("\n" + "="*70)
print("ИНФОРМАЦИЯ О ЗАГРУЖЕННЫХ ДАННЫХ")
print("="*70)
df.info()
print(f"\nПропуски в ключевых колонках:\n{df[['age','renta','antiguedad']].isnull().sum()}")

# ================================
# ДАЛЬНЕЙШИЙ EDA (как в предыдущем коде)
# ================================

# Количество продуктов на клиента
df['num_products'] = df[target_cols].sum(axis=1)
print(f"\nСреднее количество продуктов: {df['num_products'].mean():.2f}")
print(f"Медиана: {df['num_products'].median()}")
print(f"Максимум: {df['num_products'].max()}")

# Топ-15 продуктов
product_freq = df[target_cols].sum().sort_values(ascending=False)
top_products = product_freq.head(15)

# График распределения количества продуктов
fig1, ax1 = plt.subplots()
sns.histplot(df['num_products'], bins=range(0, df['num_products'].max()+2), discrete=True, ax=ax1)
ax1.set_title('Распределение количества продуктов на клиента')

# График топ-15 продуктов
fig2, ax2 = plt.subplots(figsize=(12,6))
top_products.plot(kind='bar', color='skyblue', ax=ax2)
ax2.set_title('Топ-15 наиболее распространённых продуктов')
ax2.tick_params(axis='x', rotation=45)

# Корреляция Жаккара для топ-20 продуктов (упрощённо)
top20 = product_freq.head(20).index
jaccard_mat = np.zeros((len(top20), len(top20)))
for i, p1 in enumerate(top20):
    for j, p2 in enumerate(top20):
        jaccard_mat[i, j] = (df[p1] & df[p2]).sum() / (df[p1] | df[p2]).sum()  # ускоренная версия
fig3, ax3 = plt.subplots(figsize=(12,10))
sns.heatmap(jaccard_mat, xticklabels=top20, yticklabels=top20, cmap='Blues', annot=False, ax=ax3)
ax3.set_title('Сходство Жаккара (топ-20 продуктов)')

# Временная динамика (если есть даты)
if 'fecha_dato' in df.columns and df['fecha_dato'].notna().any():
    monthly = df.groupby(df['fecha_dato'].dt.to_period('M')).agg({
        'num_products': 'mean',
        **{p: 'mean' for p in top_products.head(5).index}
    }).reset_index()
    monthly['fecha_dato'] = monthly['fecha_dato'].astype(str)
    
    fig4, ax4 = plt.subplots(figsize=(14,6))
    for prod in top_products.head(5).index:
        ax4.plot(monthly['fecha_dato'], monthly[prod], marker='o', label=prod)
    ax4.set_title('Динамика проникновения топ-5 продуктов')
    ax4.legend()
    plt.xticks(rotation=45)

# Связь возраста, дохода, стажа с продуктами
top3 = top_products.head(3).index
for prod in top3:
    fig, axes = plt.subplots(1, 3, figsize=(15,4))
    # Возраст
    axes[0].boxplot([df[df[prod]==1]['age'].dropna(), df[df[prod]==0]['age'].dropna()],
                    labels=['Есть продукт', 'Нет продукта'])
    axes[0].set_title(f'Возраст vs {prod}')
    # Доход
    axes[1].boxplot([df[df[prod]==1]['renta'].dropna(), df[df[prod]==0]['renta'].dropna()],
                    labels=['Есть', 'Нет'])
    axes[1].set_title(f'Доход vs {prod}')
    # Стаж
    axes[2].boxplot([df[df[prod]==1]['antiguedad'].dropna(), df[df[prod]==0]['antiguedad'].dropna()],
                    labels=['Есть', 'Нет'])
    axes[2].set_title(f'Стаж vs {prod}')
    plt.tight_layout()

# Категориальные признаки
categorical_cols = ['sexo', 'segmento', 'ind_empleado']
for col in categorical_cols:
    if col in df.columns:
        fig, axes = plt.subplots(1, len(top3), figsize=(15,4))
        for i, prod in enumerate(top3):
            grouped = df.groupby(col, observed=True)[prod].mean().sort_values(ascending=False)
            axes[i].barh(grouped.index.astype(str), grouped.values, color='teal')
            axes[i].set_title(f'Доля {prod} по {col}')
        plt.tight_layout()

# Корреляционная матрица
sample_corr = df[['age', 'renta', 'antiguedad', 'num_products'] + list(top3)].dropna().sample(min(30000, len(df)))
corr_matrix = sample_corr.corr()
fig, ax = plt.subplots(figsize=(8,6))
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', center=0, fmt='.2f', ax=ax)
ax.set_title('Корреляция числовых признаков и топ-продуктов')

# Сохраняем всё в PDF
with PdfPages('eda_bank_products_report.pdf') as pdf:
    for i in plt.get_fignums():
        fig = plt.figure(i)
        pdf.savefig(fig)
        plt.close(fig)

print("\n" + "="*70)
print("ОТЧЁТ СОХРАНЁН: eda_bank_products_report.pdf")
print("="*70)
print(f"Уникальных клиентов: {df['ncodpers'].nunique():,}")
print(f"Самый популярный продукт: {product_freq.index[0]} ({product_freq.iloc[0]:,} записей)")
print("Рекомендации:")
print("- Временной срез: для предсказаний использовать лаговые признаки")
print("- Заполнять пропуски renta медианой по prov или segmento")
print("- Учитывать дисбаланс классов при обучении")

ЗАГРУЗКА ДАННЫХ (СЕМПЛ 500 000 СТРОК)
Найдено целевых колонок (продуктов): 24
Загружено 500,000 строк, 48 колонок
Используемая память: 511.5 MB

ИНФОРМАЦИЯ О ЗАГРУЖЕННЫХ ДАННЫХ
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 500000 entries, 0 to 499999
Data columns (total 48 columns):
 #   Column                 Non-Null Count   Dtype         
---  ------                 --------------   -----         
 0   fecha_dato             500000 non-null  datetime64[ns]
 1   ncodpers               500000 non-null  int64         
 2   ind_empleado           494611 non-null  category      
 3   pais_residencia        494611 non-null  category      
 4   sexo                   494610 non-null  category      
 5   age                    494611 non-null  float32       
 6   fecha_alta             494611 non-null  datetime64[ns]
 7   ind_nuevo              494611 non-null  float32       
 8   antiguedad             494611 non-null  float32       
 9   indrel                 494611 non-null  float32

In [5]:
# Моделирование банковских продуктов (финальная стабильная версия)

import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

from catboost import CatBoostClassifier
from sklearn.multiclass import OneVsRestClassifier
import joblib
import gc

NA_VALUES = [' NA', '     NA', 'NA', 'n/a', 'N/A', 'NULL', 'null', '', ' ', '?', 'unknown']


def build_proba_matrix(y_pred, n_samples, n_products):
    """OneVsRest + CatBoost возвращает (n_samples, n_products), не список."""
    if isinstance(y_pred, list):
        cols = []
        for p in y_pred:
            p = np.asarray(p)
            cols.append(p[:, 1] if p.ndim == 2 and p.shape[1] == 2 else p.ravel())
        return np.column_stack(cols)

    arr = np.asarray(y_pred)
    if arr.ndim == 3:
        return arr[:, :, 1]
    if arr.ndim == 2:
        if arr.shape[0] == n_samples and arr.shape[1] == n_products:
            return arr
        if arr.shape[0] == n_products and arr.shape[1] == n_samples:
            return arr.T
    raise ValueError(f'Неожиданная форма predict_proba: {arr.shape}')


def map_at_k(y_true, y_score, k=7):
  """y_true, y_score: (n_samples, n_products)"""
  ap = []
  for i in range(y_true.shape[0]):
    true = y_true[i]
    if true.sum() == 0:
      continue
    top = np.argsort(y_score[i])[::-1][:k]
    rel = true[top]
    cum_prec = np.cumsum(rel) / (np.arange(1, len(rel) + 1))
    ap.append(np.sum(cum_prec * rel) / min(k, true.sum()))
  return np.mean(ap) if ap else 0.0


# ## 1. Загрузка данных (2016 год, 20% клиентов)
print('Загрузка данных...')
dtypes = {
    'fecha_dato': 'object', 'ncodpers': 'int32', 'age': 'float32',
    'antiguedad': 'float32', 'renta': 'float32', 'sexo': 'object',
    'segmento': 'object', 'ind_empleado': 'object', 'canal_entrada': 'object',
    'ind_nuevo': 'float32', 'indrel': 'float32',
}

sample_headers = pd.read_csv('data/train_ver2.csv', nrows=0)
target_cols = [
    col for col in sample_headers.columns
    if col.startswith('ind_') and col.endswith('_ult1')
]

usecols = [
    'fecha_dato', 'ncodpers', 'age', 'antiguedad', 'renta', 'sexo',
    'segmento', 'ind_empleado', 'canal_entrada', 'ind_nuevo', 'indrel',
] + target_cols

df = pd.read_csv(
    'data/train_ver2.csv',
    usecols=usecols,
    dtype=dtypes,
    na_values=NA_VALUES,
    skipinitialspace=True,
    low_memory=False,
)
df['fecha_dato'] = pd.to_datetime(df['fecha_dato'])

for col in target_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0).astype('int8')

df = df[df['fecha_dato'] >= '2016-01-01']
print(f'Всего записей за 2016: {len(df):,}')

np.random.seed(42)
unique_clients = df['ncodpers'].unique()
selected_clients = np.random.choice(
    unique_clients, size=int(len(unique_clients) * 0.2), replace=False,
)
df = df[df['ncodpers'].isin(selected_clients)]
print(f'После выборки 20% клиентов: {len(df):,} записей')

# ## 2. Создание таргета (новые продукты в следующем месяце)
df = df.sort_values(['ncodpers', 'fecha_dato']).reset_index(drop=True)

shifted = df.groupby('ncodpers')[target_cols].shift(-1)
for col in target_cols:
    df[f'{col}_new'] = ((shifted[col] == 1) & (df[col] == 0)).astype('int8')

new_target_cols = [f'{col}_new' for col in target_cols]
# Убираем последний месяц клиента — для него нет следующего периода
last_month = df.groupby('ncodpers')['fecha_dato'].transform('max')
df = df[df['fecha_dato'] < last_month].copy()
print(f'Осталось строк (без последнего месяца клиента): {len(df):,}')

# ## 3. Признаки и предобработка (как в EDA: renta по segmento)
feature_cols = [
    'age', 'antiguedad', 'renta', 'sexo', 'segmento', 'ind_empleado', 'canal_entrada',
]

df['renta'] = df.groupby('segmento')['renta'].transform(lambda x: x.fillna(x.median()))
df['renta'] = df['renta'].fillna(df['renta'].median())
df['age'] = df['age'].fillna(df['age'].median())
df['antiguedad'] = df['antiguedad'].fillna(0)

cat_features = ['sexo', 'segmento', 'ind_empleado', 'canal_entrada']
for col in cat_features:
    df[col] = df[col].astype(str).fillna('unknown')

# ## 4. Time-Based Split
train_mask = df['fecha_dato'].isin(['2016-01-28', '2016-02-28'])
val_mask = df['fecha_dato'] == '2016-03-28'
test_mask = df['fecha_dato'] == '2016-04-28'

X_train = df.loc[train_mask, feature_cols]
y_train = df.loc[train_mask, new_target_cols]
X_val = df.loc[val_mask, feature_cols]
y_val = df.loc[val_mask, new_target_cols]
X_test = df.loc[test_mask, feature_cols]
y_test = df.loc[test_mask, new_target_cols]

print(f'Train: {X_train.shape}, Val: {X_val.shape}, Test: {X_test.shape}')

del df
gc.collect()

# ## 5. Обучение модели
base_model = CatBoostClassifier(
    iterations=50,
    learning_rate=0.1,
    depth=4,
    auto_class_weights='Balanced',
    random_seed=42,
    verbose=10,
    cat_features=cat_features,
    loss_function='Logloss',
)

model = OneVsRestClassifier(base_model)

print('Обучение начато...')
model.fit(X_train, y_train)
print('Обучение завершено.')

joblib.dump(model, 'model.bin')
print('Модель сохранена как model.bin')

# ## 6. Оценка качества (MAP@7)
n_products = len(new_target_cols)

for split_name, X_split, y_split in [
    ('Validation', X_val, y_val),
    ('Test', X_test, y_test),
]:
    proba_matrix = build_proba_matrix(
        model.predict_proba(X_split), len(X_split), n_products,
    )
    assert proba_matrix.shape == y_split.shape, (
        f'{split_name}: proba {proba_matrix.shape} != y {y_split.shape}'
    )
    map7 = map_at_k(y_split.values, proba_matrix, k=7)
    print(f'{split_name} MAP@7: {map7:.4f}')

# ## 7. Важность признаков
# У части продуктов в train нет положительных примеров — sklearn подставляет _ConstantPredictor
trained_estimators = [
    est for est in model.estimators_
    if hasattr(est, 'feature_importances_')
]
if trained_estimators:
    avg_importance = np.mean(
        [est.feature_importances_ for est in trained_estimators], axis=0,
    )
    imp = pd.DataFrame({
        'feature': feature_cols,
        'importance': avg_importance,
    }).sort_values('importance', ascending=False)
    print(
        f'\nТоп-5 важных признаков (среднее по {len(trained_estimators)} обученным продуктам):'
    )
    print(imp.head())
else:
    print('Нет обученных CatBoost-моделей для расчёта важности признаков.')

Загрузка данных...
Всего записей за 2016: 4,621,976
После выборки 20% клиентов: 924,429 записей
Осталось строк (без последнего месяца клиента): 736,745
Train: (366632, 7), Val: (184695, 7), Test: (185418, 7)
Обучение начато...
0:	learn: 0.6764594	total: 69ms	remaining: 3.38s
10:	learn: 0.6003556	total: 515ms	remaining: 1.82s
20:	learn: 0.5744768	total: 975ms	remaining: 1.35s
30:	learn: 0.5634131	total: 1.44s	remaining: 882ms
40:	learn: 0.5581184	total: 1.94s	remaining: 427ms
49:	learn: 0.5556906	total: 2.38s	remaining: 0us
0:	learn: 0.6916017	total: 68.4ms	remaining: 3.35s
10:	learn: 0.5121648	total: 471ms	remaining: 1.67s
20:	learn: 0.3893375	total: 906ms	remaining: 1.25s
30:	learn: 0.3590947	total: 1.35s	remaining: 831ms
40:	learn: 0.3269943	total: 1.78s	remaining: 390ms
49:	learn: 0.2950514	total: 2.14s	remaining: 0us
0:	learn: 0.6827836	total: 71.2ms	remaining: 3.49s
10:	learn: 0.6275181	total: 601ms	remaining: 2.13s
20:	learn: 0.6033009	total: 1.1s	remaining: 1.52s
30:	learn: 0.59